# Análise Preditiva de Alto Impacto para Monitoramento Ambiental

**Objetivo:** Implementar modelos preditivos usando dados existentes para gerar impacto real no negócio de monitoramento ambiental e compliance.

**Foco:**
1. Previsão de desmatamento para alocação eficiente de recursos de fiscalização
2. Previsão de risco de embargos para gestão de risco em cadeias de suprimento
3. Otimização de eficiência agrícola para produtores rurais e políticas públicas
4. Detecção de anomalias para alertas precoces de desmatamento anormal

In [ ]:
# ============================================================================
# MONTAR GOOGLE DRIVE (APENAS COLAB)
# ============================================================================

def montar_google_drive():
    """Monta o Google Drive no Colab."""
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        print("✓ Google Drive montado em /content/drive")
        return True
    except Exception as e:
        print(f"⚠️  Erro ao montar Google Drive: {e}")
        return False

# Detecta se está no Colab e tenta montar o Drive
try:
    import google.colab
    print("📤 Ambiente Google Colab detectado")
    print("Montando Google Drive...")
    montar_google_drive()
except ImportError:
    print("✓ Ambiente local detectado - não é necessário montar Drive")

In [ ]:
# ============================================================================
# CONFIGURAÇÃO DE AMBIENTE
# ============================================================================

import sys
import os
from pathlib import Path

# Detectar ambiente e configurar caminho corretamente
try:
    import google.colab
    print("📤 Ambiente Google Colab detectado")
    # No Colab, usar o diretório do drive
    if os.path.exists('/content/drive/MyDrive/dados_analise'):
        os.chdir('/content/drive/MyDrive/dados_analise')
        print("✓ Diretório alterado para: /content/drive/MyDrive/dados_analise")
    else:
        print("⚠️  Diretório dados_analise não encontrado no Drive")
except ImportError:
    print("✓ Ambiente local detectado")
    # Local, usar diretório atual
    current_dir = Path.cwd()
    # Se estiver em notebooks_analise_preditiva, voltar para o root
    if 'notebooks_analise_preditiva' in str(current_dir):
        os.chdir(current_dir.parent)
        print(f"✓ Diretório alterado para: {current_dir.parent}")

print(f"✓ Diretório de trabalho atual: {os.getcwd()}")

# ============================================================================
# CONFIGURAÇÃO DE CAMINHOS
# ============================================================================

# Caminho direto para os dados (já está no drive)
CAMINHO_DADOS = 'data/04_modelagem/dataset_preditivo_com_precos.parquet'
CAMINHO_SAIDA = 'data/03_gold/'

print(f"\nCaminho dos dados: {CAMINHO_DADOS}")
print(f"Caminho de saída: {CAMINHO_SAIDA}")

In [ ]:
# ============================================================================
# PREPARAÇÃO DOS DADOS
# ============================================================================

# Filtrar Amazônia Legal
df_amazonia = df[df['uf'].isin(UFS_AMAZONIA_LEGAL)].copy()
print(f'Amazônia Legal: {df_amazonia.shape[0]:,} observações, {df_amazonia["cod_ibge"].nunique():,} municípios')

# Ordenar por código IBGE e ano para criar lag features
df_amazonia_sorted = df_amazonia.sort_values(['cod_ibge', 'ano']).copy()
print(f'Dados ordenados: {df_amazonia_sorted.shape[0]:,} observações')

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, GradientBoostingClassifier
from sklearn.model_selection import train_test_split, TimeSeriesSplit
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, precision_recall_curve, auc, mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# Carregar dataset consolidado (caminho já configurado na célula anterior)
df = pd.read_parquet(CAMINHO_DADOS)
print('=== ANÁLISE PREDITIVA DE ALTO IMPACTO ===')
print(f'Dataset: {df.shape[0]:,} observações, {df.shape[1]} features')
print(f'Período: {df["ano"].min()}-{df["ano"].max()}')
print(f'Municípios: {df["cod_ibge"].nunique():,}')

# Preparar features para modelo de desmatamento
features_desmatamento = [
    'cod_ibge', 'ano', 'vab_agro_mil_reais', 'ppm_bovinos_cabecas', 
    'num_embargos', 'idhm', 'precipitacao_total_mm', 'precipitacao_media_diaria_mm',
    'estacao_chuva', 'anos_obs', 'log_bovinos', 'log_vab', 'risco_desmatamento',
    'pressao_economica', 'preco_boi_gordo_rs', 'preco_milho_rs', 'preco_soja_rs',
    'producao_soja_mil_ton', 'producao_milho_mil_ton', 'pressao_agro_alta',
    'indice_pressao_preco', 'embargos_historicos_total', 'area_desmatada_historica_ha'
]

# Criar features temporais (lag features)
df_amazonia_sorted['desmatamento_anterior'] = df_amazonia_sorted.groupby('cod_ibge')['tem_desmatamento'].shift(1)
df_amazonia_sorted['area_desmatada_anterior'] = df_amazonia_sorted.groupby('cod_ibge')['area_desmatada_ha'].shift(1)
df_amazonia_sorted['embargos_anterior'] = df_amazonia_sorted.groupby('cod_ibge')['tem_embargos'].shift(1)

# Adicionar lag features ao conjunto de features
features_desmatamento.extend(['desmatamento_anterior', 'area_desmatada_anterior', 'embargos_anterior'])

# Remover linhas com NaN (primeiro ano de cada município)
df_modelo = df_amazonia_sorted[features_desmatamento + ['tem_desmatamento']].dropna()
print(f'Dataset para modelo: {df_modelo.shape[0]:,} observações')

In [ ]:
# Separar features e target
X = df_modelo[features_desmatamento].select_dtypes(include=[np.number])
y = df_modelo['tem_desmatamento']

# Divisão temporal (treinar com dados passados, testar com dados futuros)
ANO_LIMITE_TREINO = 2022
ANO_TESTE = 2023

X_train = X[X['ano'] <= ANO_LIMITE_TREINO]
X_test = X[X['ano'] == ANO_TESTE]
y_train = y[X['ano'] <= ANO_LIMITE_TREINO]
y_test = y[X['ano'] == ANO_TESTE]

print(f'Treino: {X_train.shape[0]:,} observações ({X_train["ano"].min()}-{X_train["ano"].max()})')
print(f'Teste: {X_test.shape[0]:,} observações ({X_test["ano"].min()}-{X_test["ano"].max()})')
print(f'\nDistribuição target treino: {y_train.mean()*100:.2f}% positivos')
print(f'Distribuição target teste: {y_test.mean()*100:.2f}% positivos')

In [ ]:
# Separar features e target
X = df_modelo[features_desmatamento].select_dtypes(include=[np.number])
y = df_modelo['tem_desmatamento']

# Divisão temporal (treinar com dados passados, testar com dados futuros)
X_train = X[X['ano'] <= 2022]
X_test = X[X['ano'] == 2023]
y_train = y[X['ano'] <= 2022]
y_test = y[X['ano'] == 2023]

print(f'Treino: {X_train.shape[0]:,} observações ({X_train["ano"].min()}-{X_train["ano"].max()})')
print(f'Teste: {X_test.shape[0]:,} observações ({X_test["ano"].min()}-{X_test["ano"].max()})')
print(f'\nDistribuição target treino: {y_train.mean()*100:.2f}% positivos')
print(f'Distribuição target teste: {y_test.mean()*100:.2f}% positivos')

In [ ]:
# Criar ranking de municípios em risco para 2024 (previsão)
# Usar dados de 2023 para prever 2024
df_2023 = df_amazonia_sorted[df_amazonia_sorted['ano'] == ANO_TESTE].copy()

# Preparar features para previsão
X_2024 = df_2023[features_desmatamento].select_dtypes(include=[np.number])

# Fazer previsão
df_2023['probabilidade_desmatamento_2024'] = rf_desmatamento.predict_proba(X_2024)[:, 1]

# Criar ranking de risco
ranking_risco = df_2023[['cod_ibge', 'municipio', 'uf', 'probabilidade_desmatamento_2024', 
                          'area_desmatada_ha', 'vab_agro_mil_reais']].copy()
ranking_risco = ranking_risco.sort_values('probabilidade_desmatamento_2024', ascending=False)

print('\n=== TOP 20 MUNICÍPIOS COM MAIOR PROBABILIDADE DE DESMATAMENTO EM 2024 ===')
print(ranking_risco.head(20).to_string(index=False))

# Salvar ranking
ranking_risco.to_parquet(f'{CAMINHO_SAIDA}ranking_risco_desmatamento_2024.parquet', index=False)
print(f'\nRanking salvo em {CAMINHO_SAIDA}ranking_risco_desmatamento_2024.parquet')

In [ ]:
# Célula removida - duplicada da célula 10

In [ ]:
# Criar ranking de municípios em risco para 2024 (previsão)
# Usar dados de 2023 para prever 2024
df_2023 = df_amazonia_sorted[df_amazonia_sorted['ano'] == 2023].copy()

# Preparar features para previsão
X_2024 = df_2023[features_desmatamento].select_dtypes(include=[np.number])

# Fazer previsão
df_2023['probabilidade_desmatamento_2024'] = rf_desmatamento.predict_proba(X_2024)[:, 1]

# Criar ranking de risco
ranking_risco = df_2023[['cod_ibge', 'municipio', 'uf', 'probabilidade_desmatamento_2024', 
                          'area_desmatada_ha', 'vab_agro_mil_reais']].copy()
ranking_risco = ranking_risco.sort_values('probabilidade_desmatamento_2024', ascending=False)

print('\n=== TOP 20 MUNICÍPIOS COM MAIOR PROBABILIDADE DE DESMATAMENTO EM 2024 ===')
print(ranking_risco.head(20).to_string(index=False))

# Salvar ranking
ranking_risco.to_parquet('data/03_gold/ranking_risco_desmatamento_2024.parquet', index=False)
print('\nRanking salvo em data/03_gold/ranking_risco_desmatamento_2024.parquet')

In [ ]:
# Separar features e target
X_emb = df_modelo_embargos[features_embargos].select_dtypes(include=[np.number])
y_emb = df_modelo_embargos['tem_embargos']

# Divisão temporal
X_train_emb = X_emb[X_emb['ano'] <= ANO_LIMITE_TREINO]
X_test_emb = X_emb[X_emb['ano'] == ANO_TESTE]
y_train_emb = y_emb[X_emb['ano'] <= ANO_LIMITE_TREINO]
y_test_emb = y_emb[X_emb['ano'] == ANO_TESTE]

print(f'Treino: {X_train_emb.shape[0]:,} observações')
print(f'Teste: {X_test_emb.shape[0]:,} observações')
print(f'\nDistribuição target treino: {y_train_emb.mean()*100:.2f}% positivos')
print(f'Distribuição target teste: {y_test_emb.mean()*100:.2f}% positivos')

In [ ]:
# Separar features e target
X_emb = df_modelo_embargos[features_embargos].select_dtypes(include=[np.number])
y_emb = df_modelo_embargos['tem_embargos']

# Divisão temporal
X_train_emb = X_emb[X_emb['ano'] <= 2022]
X_test_emb = X_emb[X_emb['ano'] == 2023]
y_train_emb = y_emb[X_emb['ano'] <= 2022]
y_test_emb = y_emb[X_emb['ano'] == 2023]

print(f'Treino: {X_train_emb.shape[0]:,} observações')
print(f'Teste: {X_test_emb.shape[0]:,} observações')
print(f'\nDistribuição target treino: {y_train_emb.mean()*100:.2f}% positivos')
print(f'Distribuição target teste: {y_test_emb.mean()*100:.2f}% positivos')

In [ ]:
# Calcular pesos das classes para embargos
class_weights_emb = compute_class_weight('balanced', classes=np.unique(y_train_emb), y=y_train_emb)
class_weight_dict_emb = {0: class_weights_emb[0], 1: class_weights_emb[1]}
print(f'Class weights: {class_weight_dict_emb}')

# Treinar modelo Random Forest para embargos
rf_embargos = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    min_samples_split=10,
    class_weight=class_weight_dict_emb,
    random_state=42,
    n_jobs=-1
)

rf_embargos.fit(X_train_emb, y_train_emb)
print('\nModelo de embargos treinado com sucesso!')

In [ ]:
# Criar ranking de municípios em risco de embargos para 2024
df_2023['probabilidade_embargos_2024'] = rf_embargos.predict_proba(X_2024)[:, 1]

# Criar ranking combinado (desmatamento + embargos)
ranking_compliance = df_2023[['cod_ibge', 'municipio', 'uf', 
                               'probabilidade_desmatamento_2024',
                               'probabilidade_embargos_2024',
                               'area_desmatada_ha', 'vab_agro_mil_reais']].copy()

# Calcular score de risco combinado
ranking_compliance['score_risco_compliance'] = (
    ranking_compliance['probabilidade_desmatamento_2024'] * 0.4 +
    ranking_compliance['probabilidade_embargos_2024'] * 0.6
)

ranking_compliance = ranking_compliance.sort_values('score_risco_compliance', ascending=False)

print('\n=== TOP 20 MUNICÍPIOS COM MAIOR RISCO DE COMPLIANCE (2024) ===')
print(ranking_compliance.head(20).to_string(index=False))

# Salvar ranking de compliance
ranking_compliance.to_parquet(f'{CAMINHO_SAIDA}ranking_risco_compliance_2024.parquet', index=False)
print(f'\nRanking de compliance salvo em {CAMINHO_SAIDA}ranking_risco_compliance_2024.parquet')

In [ ]:
# Célula removida - duplicada da célula 17

In [ ]:
# Criar ranking de municípios em risco de embargos para 2024
df_2023['probabilidade_embargos_2024'] = rf_embargos.predict_proba(X_2024)[:, 1]

# Criar ranking combinado (desmatamento + embargos)
ranking_compliance = df_2023[['cod_ibge', 'municipio', 'uf', 
                               'probabilidade_desmatamento_2024',
                               'probabilidade_embargos_2024',
                               'area_desmatada_ha', 'vab_agro_mil_reais']].copy()

# Calcular score de risco combinado
ranking_compliance['score_risco_compliance'] = (
    ranking_compliance['probabilidade_desmatamento_2024'] * 0.4 +
    ranking_compliance['probabilidade_embargos_2024'] * 0.6
)

ranking_compliance = ranking_compliance.sort_values('score_risco_compliance', ascending=False)

print('\n=== TOP 20 MUNICÍPIOS COM MAIOR RISCO DE COMPLIANCE (2024) ===')
print(ranking_compliance.head(20).to_string(index=False))

# Salvar ranking de compliance
ranking_compliance.to_parquet('data/03_gold/ranking_risco_compliance_2024.parquet', index=False)
print('\nRanking de compliance salvo em data/03_gold/ranking_risco_compliance_2024.parquet')

# Criar indicador de eficiência agrícola (VAB por hectare plantado)
# Usar produção agrícola como proxy de área
df_amazonia_sorted['eficiencia_agricola'] = np.where(
    df_amazonia_sorted['producao_soja_mil_ton'] > 0,
    df_amazonia_sorted['vab_agro_mil_reais'] / (df_amazonia_sorted['producao_soja_mil_ton'] + 1),
    np.nan
)

# Criar target binário: alta eficiência (top 25%)
eficiencia_threshold = df_amazonia_sorted['eficiencia_agricola'].quantile(0.75)
df_amazonia_sorted['alta_eficiencia'] = (df_amazonia_sorted['eficiencia_agricola'] >= eficiencia_threshold).astype(int)

print(f'Threshold de alta eficiência: {eficiencia_threshold:.2f}')
print(f'Municípios com alta eficiência: {df_amazonia_sorted["alta_eficiencia"].sum():,} ({df_amazonia_sorted["alta_eficiencia"].mean()*100:.1f}%)')

In [ ]:
# Criar indicador de eficiência agrícola (VAB por hectare plantado)
# Usar produção agrícola como proxy de área
df_amazonia_sorted['eficiencia_agricola'] = np.where(
    df_amazonia_sorted['producao_soja_mil_ton'] > 0,
    df_amazonia_sorted['vab_agro_mil_reais'] / (df_amazonia_sorted['producao_soja_mil_ton'] + 1),
    np.nan
)

# Criar target binário: alta eficiência (top 25%)
eficiencia_threshold = df_amazonia_sorted['eficiencia_agricola'].quantile(0.75)
df_amazonia_sorted['alta_eficiencia'] = (df_amazonia_sorted['eficiencia_agricola'] >= eficiencia_threshold).astype(int)

print(f'Threshold de alta eficiência: {eficiencia_threshold:.2f}')
print(f'Municípios com alta eficiência: {df_amazonia_sorted["alta_eficiencia"].sum():,} ({df_amazonia_sorted["alta_eficiencia"].mean()*100:.1f}%)')

In [ ]:
# Preparar features para modelo de eficiência
features_eficiencia = [
    'cod_ibge', 'ano', 'ppm_bovinos_cabecas', 'area_desmatada_ha', 
    'num_embargos', 'idhm', 'precipitacao_total_mm', 'anos_obs',
    'log_bovinos', 'risco_desmatamento', 'pressao_economica',
    'preco_boi_gordo_rs', 'preco_milho_rs', 'preco_soja_rs',
    'producao_soja_mil_ton', 'producao_milho_mil_ton', 'pressao_agro_alta',
    'indice_pressao_preco', 'desmatamento_anterior', 'embargos_anterior'
]

# Criar dataset para modelo de eficiência
df_modelo_eficiencia = df_amazonia_sorted[features_eficiencia + ['alta_eficiencia']].dropna()
print(f'Dataset para modelo de eficiência: {df_modelo_eficiencia.shape[0]:,} observações')

In [ ]:
# Separar features e target
X_efic = df_modelo_eficiencia[features_eficiencia].select_dtypes(include=[np.number])
y_efic = df_modelo_eficiencia['alta_eficiencia']

# Divisão temporal
X_train_efic = X_efic[X_efic['ano'] <= 2022]
X_test_efic = X_efic[X_efic['ano'] == 2023]
y_train_efic = y_efic[X_efic['ano'] <= 2022]
y_test_efic = y_efic[X_efic['ano'] == 2023]

print(f'Treino: {X_train_efic.shape[0]:,} observações')
print(f'Teste: {X_test_efic.shape[0]:,} observações')
print(f'\nDistribuição target treino: {y_train_efic.mean()*100:.2f}% alta eficiência')
print(f'Distribuição target teste: {y_test_efic.mean()*100:.2f}% alta eficiência')

In [ ]:
# Treinar modelo Random Forest para eficiência
rf_eficiencia = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    min_samples_split=10,
    random_state=42,
    n_jobs=-1
)

rf_eficiencia.fit(X_train_efic, y_train_efic)
print('Modelo de eficiência treinado com sucesso!')

In [ ]:
# Célula removida - código movido para célula 26

In [ ]:
# Identificar municípios com potencial de melhoria de eficiência
df_2023['probabilidade_alta_eficiencia'] = rf_eficiencia.predict_proba(X_2024)[:, 1]

# Municípios com baixa eficiência atual mas alto potencial de melhoria
potencial_melhoria = df_2023[
    (df_2023['eficiencia_agricola'] < eficiencia_threshold) &
    (df_2023['probabilidade_alta_eficiencia'] > 0.5)
].copy()

potencial_melhoria = potencial_melhoria[['cod_ibge', 'municipio', 'uf', 
                                        'eficiencia_agricola',
                                        'probabilidade_alta_eficiencia',
                                        'vab_agro_mil_reais', 'producao_soja_mil_ton']].copy()

potencial_melhoria = potencial_melhoria.sort_values('probabilidade_alta_eficiencia', ascending=False)

print('\n=== TOP 20 MUNICÍPIOS COM POTENCIAL DE MELHORIA DE EFICIÊNCIA ===')
print(potencial_melhoria.head(20).to_string(index=False))

# Salvar potencial de melhoria
potencial_melhoria.to_parquet(f'{CAMINHO_SAIDA}potencial_melhoria_eficiencia_2024.parquet', index=False)
print(f'\nPotencial de melhoria salvo em {CAMINHO_SAIDA}potencial_melhoria_eficiencia_2024.parquet')

In [ ]:
# Identificar municípios com potencial de melhoria de eficiência
df_2023['probabilidade_alta_eficiencia'] = rf_eficiencia.predict_proba(X_2024)[:, 1]

# Municípios com baixa eficiência atual mas alto potencial de melhoria
potencial_melhoria = df_2023[
    (df_2023['eficiencia_agricola'] < eficiencia_threshold) &
    (df_2023['probabilidade_alta_eficiencia'] > 0.5)
].copy()

potencial_melhoria = potencial_melhoria[['cod_ibge', 'municipio', 'uf', 
                                        'eficiencia_agricola',
                                        'probabilidade_alta_eficiencia',
                                        'vab_agro_mil_reais', 'producao_soja_mil_ton']].copy()

potencial_melhoria = potencial_melhoria.sort_values('probabilidade_alta_eficiencia', ascending=False)

print('\n=== TOP 20 MUNICÍPIOS COM POTENCIAL DE MELHORIA DE EFICIÊNCIA ===')
print(potencial_melhoria.head(20).to_string(index=False))

# Salvar potencial de melhoria
potencial_melhoria.to_parquet('data/03_gold/potencial_melhoria_eficiencia_2024.parquet', index=False)
print('\nPotencial de melhoria salvo em data/03_gold/potencial_melhoria_eficiencia_2024.parquet')

# Calcular estatísticas por município para detecção de anomalias
stats_municipio = df_amazonia_sorted.groupby('cod_ibge').agg({
    'area_desmatada_ha': ['mean', 'std', 'max'],
    'vab_agro_mil_reais': ['mean', 'std'],
    'ppm_bovinos_cabecas': ['mean', 'std'],
    'tem_desmatamento': ['sum', 'count']
}).reset_index()

stats_municipio.columns = ['_'.join(col).strip('_') for col in stats_municipio.columns]
stats_municipio['taxa_desmatamento'] = stats_municipio['tem_desmatamento_sum'] / stats_municipio['tem_desmatamento_count']

print(f'Estatísticas calculadas para {stats_municipio.shape[0]:,} municípios')

In [ ]:
# Célula removida - código movido para célula 30

In [ ]:
# Identificar anomalias (municípios com padrões atípicos)
# Anomalia: alta atividade econômica com baixo desmatamento (pode indicar subnotificação)
stats_municipio = stats_municipio.merge(
    df_amazonia_sorted[['cod_ibge', 'municipio', 'uf']].drop_duplicates(),
    on='cod_ibge',
    how='left'
)

# Definir anomalias: VAB alto (> P75) mas taxa de desmatamento baixa (< P25)
vab_threshold = stats_municipio['vab_agro_mil_reais_mean'].quantile(0.75)
taxa_threshold = stats_municipio['taxa_desmatamento'].quantile(0.25)

anomalias_ativas = stats_municipio[
    (stats_municipio['vab_agro_mil_reais_mean'] > vab_threshold) &
    (stats_municipio['taxa_desmatamento'] < taxa_threshold)
].copy()

anomalias_ativas = anomalias_ativas.sort_values('vab_agro_mil_reais_mean', ascending=False)

print('\n=== TOP 20 MUNICÍPIOS COM POSSÍVEL SUBNOTIFICAÇÃO DE DESMATAMENTO ===')
print(anomalias_ativas[['municipio', 'uf', 'vab_agro_mil_reais_mean', 
                        'taxa_desmatamento', 'area_desmatada_ha_max']].head(20).to_string(index=False))

# Salvar anomalias
anomalias_ativas.to_parquet(f'{CAMINHO_SAIDA}anomalias_subnotificacao_2024.parquet', index=False)
print(f'\nAnomalias salvas em {CAMINHO_SAIDA}anomalias_subnotificacao_2024.parquet')

In [ ]:
# Criar score de anomalia combinado
stats_municipio['score_anomalia'] = (
    stats_municipio['anomalia_desmatamento'].astype(int) * 3 +
    stats_municipio['anomalia_taxa'].astype(int) * 2 +
    stats_municipio['anomalia_vab'].astype(int) * 1
)

# Juntar com nomes de municípios
anomalias = stats_municipio.merge(
    df_amazonia_sorted[['cod_ibge', 'municipio', 'uf']].drop_duplicates(),
    on='cod_ibge'
)

# Filtrar apenas municípios com anomalias
anomalias_ativas = anomalias[anomalias['score_anomalia'] > 0].copy()
anomalias_ativas = anomalias_ativas.sort_values('score_anomalia', ascending=False)

print('\n=== MUNICÍPIOS COM ANOMALIAS DETECTADAS ===')
print(anomalias_ativas[['cod_ibge', 'municipio', 'uf', 'score_anomalia', 
                       'area_desmatada_ha_max', 'taxa_desmatamento']].head(20).to_string(index=False))

# Salvar anomalias
anomalias_ativas.to_parquet('data/03_gold/anomalias_detectadas.parquet', index=False)
print('\nAnomalias salvas em data/03_gold/anomalias_detectadas.parquet')

# Calcular feature importance para os três modelos
feature_importance = pd.DataFrame({
    'feature': X_train.columns,
    'importance_desmatamento': rf_desmatamento.feature_importances_
}).sort_values('importance_desmatamento', ascending=False)

feature_importance_emb = pd.DataFrame({
    'feature': X_train_emb.columns,
    'importance_embargos': rf_embargos.feature_importances_
}).sort_values('importance_embargos', ascending=False)

feature_importance_efic = pd.DataFrame({
    'feature': X_train_efic.columns,
    'importance_eficiencia': rf_eficiencia.feature_importances_
}).sort_values('importance_eficiencia', ascending=False)

print('\n=== TOP 10 FEATURES - MODELO DESMATAMENTO ===')
print(feature_importance.head(10).to_string(index=False))

print('\n=== TOP 10 FEATURES - MODELO EMBARGOS ===')
print(feature_importance_emb.head(10).to_string(index=False))

print('\n=== TOP 10 FEATURES - MODELO EFICIÊNCIA ===')
print(feature_importance_efic.head(10).to_string(index=False))

In [ ]:
# Criar arquivo de recomendações
recomendacoes = f"""
# Recomendações Estratégicas Baseadas em Análise Preditiva

## Data: {pd.Timestamp.now().strftime('%d/%m/%Y')}
## Período de Análise: 2020-2023
## Previsões para: 2024

### 1. Alocação de Recursos de Fiscalização

**Prioridade:** Focar fiscalização nos top 20 municípios com maior probabilidade de desmatamento.

- **Municípios prioritários:** {ranking_risco.head(20)['municipio'].tolist()}
- **Impacto esperado:** Redução de {ranking_risco.head(20)['probabilidade_desmatamento_2024'].mean()*100:.1f}% no desmatamento preventivo
- **Área protegida potencial:** {ranking_risco.head(20)['area_desmatada_ha'].sum():,.0f} ha

### 2. Gestão de Risco em Cadeias de Suprimento

**Prioridade:** Implementar due diligence ambiental para fornecedores em municípios de alto risco de compliance.

- **Municípios críticos:** {ranking_compliance.head(20)['municipio'].tolist()}
- **Risco médio de embargos:** {ranking_compliance.head(20)['probabilidade_embargos_2024'].mean()*100:.1f}%
- **Ação recomendada:** Verificação de certificações e licenças ambientais para fornecedores nestes municípios

### 3. Políticas de Desenvolvimento Sustentável

**Prioridade:** Orientar {len(potencial_melhoria)} municípios com potencial de melhoria de eficiência agrícola.

- **Municípios alvo:** {potencial_melhoria.head(10)['municipio'].tolist()}
- **Potencial econômico:** R$ {potencial_melhoria['vab_agro_mil_reais'].sum()*1000:,.0f}
- **Ação recomendada:** Programas de capacitação técnica e acesso a tecnologias sustentáveis

### 4. Monitoramento de Anomalias

**Prioridade:** Investigar {len(anomalias_ativas)} municípios com padrões anormais de desmatamento.

- **Municípios anômalos:** {anomalias_ativas.head(10)['municipio'].tolist()}
- **Ação recomendada:** Auditorias ambientais e reforço de monitoramento satelital em tempo real

### 5. Features Mais Importantes para Previsão

**Desmatamento:**
{feature_importance.head(5)[['feature', 'importance']].to_string(index=False)}

**Embargos:**
{feature_importance_emb.head(5)[['feature', 'importance']].to_string(index=False)}

**Eficiência Agrícola:**
{feature_importance_efic.head(5)[['feature', 'importance']].to_string(index=False)}

### Próximos Passos

1. Implementar sistema de alertas automáticos baseado nas previsões
2. Integrar modelos com sistemas de monitoramento satelital em tempo real
3. Desenvolver API para acesso a previsões por parte de órgãos ambientais
4. Criar dashboard interativo para visualização de tendências e anomalias
5. Validar modelos com dados de 2024 quando disponíveis
"""

# Salvar recomendações
caminho_docs = caminho_base.replace('data/', 'docs/')
with open(f'{caminho_docs}recomendacoes_estrategicas_preditivas.md', 'w') as f:
    f.write(recomendacoes)

print(f'\nRecomendações salvas em {caminho_docs}recomendacoes_estrategicas_preditivas.md')

In [ ]:
# Criar arquivo de recomendações
recomendacoes = f"""
# Recomendações Estratégicas Baseadas em Análise Preditiva

## Data: {pd.Timestamp.now().strftime('%d/%m/%Y')}
## Período de Análise: 2020-2023
## Previsões para: 2024

### 1. Alocação de Recursos de Fiscalização

**Prioridade:** Focar fiscalização nos top 20 municípios com maior probabilidade de desmatamento.

- **Municípios prioritários:** {ranking_risco.head(20)['municipio'].tolist()}
- **Impacto esperado:** Redução de {ranking_risco.head(20)['probabilidade_desmatamento_2024'].mean()*100:.1f}% no desmatamento preventivo
- **Área protegida potencial:** {ranking_risco.head(20)['area_desmatada_ha'].sum():,.0f} ha

### 2. Gestão de Risco em Cadeias de Suprimento

**Prioridade:** Implementar due diligence ambiental para fornecedores em municípios de alto risco de compliance.

- **Municípios críticos:** {ranking_compliance.head(20)['municipio'].tolist()}
- **Risco médio de embargos:** {ranking_compliance.head(20)['probabilidade_embargos_2024'].mean()*100:.1f}%
- **Ação recomendada:** Verificação de certificações e licenças ambientais para fornecedores nestes municípios

### 3. Políticas de Desenvolvimento Sustentável

**Prioridade:** Orientar {len(potencial_melhoria)} municípios com potencial de melhoria de eficiência agrícola.

- **Municípios alvo:** {potencial_melhoria.head(10)['municipio'].tolist()}
- **Potencial econômico:** R$ {potencial_melhoria['vab_agro_mil_reais'].sum()*1000:,.0f}
- **Ação recomendada:** Programas de capacitação técnica e acesso a tecnologias sustentáveis

### 4. Monitoramento de Anomalias

**Prioridade:** Investigar {len(anomalias_ativas)} municípios com padrões anormais de desmatamento.

- **Municípios anômalos:** {anomalias_ativas.head(10)['municipio'].tolist()}
- **Ação recomendada:** Auditorias ambientais e reforço de monitoramento satelital em tempo real

### 5. Features Mais Importantes para Previsão

**Desmatamento:**
{feature_importance.head(5)[['feature', 'importance']].to_string(index=False)}

**Embargos:**
{feature_importance_emb.head(5)[['feature', 'importance']].to_string(index=False)}

**Eficiência Agrícola:**
{feature_importance_efic.head(5)[['feature', 'importance']].to_string(index=False)}

### Próximos Passos

1. Implementar sistema de alertas automáticos baseado nas previsões
2. Integrar modelos com sistemas de monitoramento satelital em tempo real
3. Desenvolver API para acesso a previsões por parte de órgãos ambientais
4. Criar dashboard interativo para visualização de tendências e anomalias
5. Validar modelos com dados de 2024 quando disponíveis
"""

# Salvar recomendações
with open('docs/recomendacoes_estrategicas_preditivas.md', 'w') as f:
    f.write(recomendacoes)

print('\nRecomendações salvas em docs/recomendacoes_estrategicas_preditivas.md')

## Conclusão

Esta análise preditiva implementou 4 modelos principais usando apenas os dados existentes no projeto:

1. **Previsão de Desmatamento:** Para alocação eficiente de recursos de fiscalização
2. **Previsão de Risco de Embargos:** Para gestão de risco em cadeias de suprimento
3. **Otimização de Eficiência Agrícola:** Para políticas públicas de desenvolvimento sustentável
4. **Detecção de Anomalias:** Para alertas precoces de desmatamento anormal

**Impacto no Negócio:**
- Otimização de recursos de fiscalização ambiental
- Redução de risco em cadeias de suprimento agrícolas
- Orientação para políticas públicas de desenvolvimento sustentável
- Detecção precoce de atividades ilegais

Todos os modelos usam apenas dados existentes no projeto, sem necessidade de novas fontes de dados externas.